# STEP 2 (v3) — Rx-to-OTC Cross-Sell, BNF-Class Level, Exact Pairwise Computation  
==================================================================================  
Rebuilt per Mustafa Ghafouri's data-access notebook (Fidan_CrossSell_Data_  
Access_2026-08-21), which independently diagnosed the two real problems in  
the previous FP-Growth version and gave a cleaner architecture:  
  
  PROBLEM 1 (why FP-Growth kept needing sampling/timeouts/OOM workarounds):  
  we were attacking the wrong axis. The one-hot matrix FP-Growth needs is  
  baskets x items. We fixed the ITEM count at thousands of individual SKUs  
  (ProductKey) and only ever cut the BASKET axis (sampling). Mustafa's  
  point: the item axis costs nothing to cut — aggregating individual  
  products to their BNF class (chapter+section) turns "thousands of SKUs"  
  into "~80-300 classes", a ~1,000x reduction in matrix size, with NO  
  sampling needed at all.  
  
  PROBLEM 2 (a genuine bug, not a design choice): "your 30-day forward  
  windows overlap. A patient prescribed on day 0 and day 20 has a day-25  
  OTC purchase counted in BOTH baskets, inflating support — worst for  
  frequent orderers. Attribute each OTC purchase to the nearest preceding  
  prescription only." Fixed here using pd.merge_asof(direction='backward',  
  tolerance=WINDOW_DAYS) — verified against the exact scenario Mustafa  
  described, plus window-boundary and no-preceding-anchor edge cases.  
  
ARCHITECTURE CHANGE: since we only need Rx -> OTC PAIRS (not arbitrary-  
length itemsets), this drops mlxtend/FP-Growth entirely in favour of an  
exact groupby computation — "a groupby is exact on the full population  
with no memory ceiling" (Mustafa, Cell 38). This closes the open question  
from the previous version about whether more compute/sampling would ever  
find more diversity: with no memory ceiling, the answer no longer depends  
on sample size at all — the full population is used directly.  
  
Threshold: absolute minimum count n_ab >= MIN_COUNT (Mustafa suggests 50)  
applied BEFORE computing lift, replacing the old min_support-as-fraction  
sweep entirely — no sample-size dependence left to tune.  
  
Input: clean_transactions_all_drugs.parquet (Step 1), otc_product_universe.parquet  
Output: bnf_class_otc_rules.parquet / .csv


In [1]:
import pandas as pd
import numpy as np

### CONFIG

In [2]:
WINDOW_DAYS = 30            # forward-only lookahead per NHS anchor (unchanged)
MIN_COUNT = 50               # Mustafa: "filter n_ab >= 50 BEFORE computing lift"
OTC_LEGAL_CATS = {"GSL", "P"}

# BNF class = Chapter + Section (ProductCodeLevel1 + ProductCodeLevel2).
# Chapter alone (~20 values) is too coarse (loses clinically meaningful
# distinctions within e.g. cardiovascular); Chapter+Section+Paragraph would
# push back toward SKU-level granularity and lose the density benefit.
# Chapter+Section matches Mustafa's estimate of "hundreds of classes" on
# the Rx side.
BNF_CLASS_COLS = ["ProductCodeLevel1", "ProductCodeLevel2"]

### PART A — Load Step 1's outputs

In [3]:
transactions = pd.read_parquet("clean_transactions_all_drugs.parquet")
otc_products = pd.read_parquet("otc_product_universe.parquet")

transactions["dispense_date_created"] = pd.to_datetime(transactions["dispense_date_created"])
print(f"[A] Loaded {len(transactions):,} transaction lines, "
      f"{transactions['CustomerKey'].nunique():,} customers")

# Gotcha: some products have no BNF chapter/section classification at all
# (ProductCodeLevel1/2 are null — e.g. unmapped/"Missing Product" rows seen
# in Dim.Product earlier). str(None) + str(None) = "NoneNone", which then
# silently becomes a fake pseudo-category that can rank as a top "rule" —
# exactly what happened in the first run of this script. Drop these rows
# BEFORE building the class label, not after.
n_before = len(transactions)
transactions = transactions.dropna(subset=["ProductCodeLevel1", "ProductCodeLevel2"])
transactions = transactions[
    (transactions["ProductCodeLevel1"].astype(str).str.strip() != "")
    & (transactions["ProductCodeLevel2"].astype(str).str.strip() != "")
]
n_after = len(transactions)
print(f"[A] Dropped {n_before - n_after:,} rows ({(n_before - n_after) / n_before:.1%}) "
      f"with missing/blank BNF chapter or section classification")

# Build the BNF class label once, reused for both NHS and OTC sides
transactions["bnf_class"] = (
    transactions["ProductCodeLevel1"].astype(str) + transactions["ProductCodeLevel2"].astype(str)
)

[A] Loaded 67,124,000 transaction lines, 1,872,959 customers
[A] Dropped 12,164,423 rows (18.1%) with missing/blank BNF chapter or section classification


### PART B — Split into NHS (POM) and OTC (GSL/P) sides, at BNF-class grain

In [4]:
nhs_df = transactions[transactions["LEGAL_CAT"] == "POM"][
    ["CustomerKey", "dispense_date_created", "bnf_class"]
].rename(columns={"dispense_date_created": "nhs_date", "bnf_class": "nhs_bnf_class"})
nhs_df = nhs_df.drop_duplicates()
nhs_df["anchor_id"] = np.arange(len(nhs_df))

otc_df = transactions[transactions["LEGAL_CAT"].isin(OTC_LEGAL_CATS)][
    ["CustomerKey", "dispense_date_created", "bnf_class"]
].rename(columns={"dispense_date_created": "otc_date", "bnf_class": "otc_bnf_class"})
otc_df = otc_df.drop_duplicates()

print(f"[B] {len(nhs_df):,} distinct NHS (customer, date, BNF class) anchors")
print(f"[B] {len(otc_df):,} distinct OTC (customer, date, BNF class) events")
print(f"[B] {nhs_df['nhs_bnf_class'].nunique():,} distinct NHS BNF classes, "
      f"{otc_df['otc_bnf_class'].nunique():,} distinct OTC BNF classes")

[B] 49,008,230 distinct NHS (customer, date, BNF class) anchors
[B] 3,556,560 distinct OTC (customer, date, BNF class) events
[B] 95 distinct NHS BNF classes, 64 distinct OTC BNF classes


### PART C — Attribute each OTC event to its NEAREST PRECEDING NHS anchor only (Mustafa's bug fix — replaces the old "every overlapping window" logic)

In [5]:
attributed = pd.merge_asof(
    otc_df.sort_values("otc_date"),
    nhs_df.sort_values("nhs_date"),
    left_on="otc_date", right_on="nhs_date",
    by="CustomerKey", direction="backward",
    tolerance=pd.Timedelta(days=WINDOW_DAYS),
)
attributed = attributed.dropna(subset=["anchor_id"])
attributed["anchor_id"] = attributed["anchor_id"].astype(nhs_df["anchor_id"].dtype)

print(f"[C] {len(attributed):,} OTC events attributed to a unique nearest-preceding "
      f"NHS anchor within {WINDOW_DAYS} days (each OTC event now attributes to "
      f"exactly ONE anchor, not every overlapping one)")

[C] 2,995,378 OTC events attributed to a unique nearest-preceding NHS anchor within 30 days (each OTC event now attributes to exactly ONE anchor, not every overlapping one)


### PART D — Exact pairwise support / confidence / lift (no FP-Growth needed — we only need Rx -> OTC pairs, not arbitrary-length itemsets)

In [6]:
n_total_baskets = nhs_df["anchor_id"].nunique()

nhs_counts = nhs_df.groupby("nhs_bnf_class")["anchor_id"].nunique()
otc_counts = (
    attributed.drop_duplicates(["anchor_id", "otc_bnf_class"])
    .groupby("otc_bnf_class")["anchor_id"].nunique()
)
pair_counts = (
    attributed.drop_duplicates(["anchor_id", "nhs_bnf_class", "otc_bnf_class"])
    .groupby(["nhs_bnf_class", "otc_bnf_class"])["anchor_id"].nunique()
    .rename("n_ab")
    .reset_index()
)

print(f"[D] {len(pair_counts):,} raw (NHS class, OTC class) pairs found "
      f"(before MIN_COUNT filter)")

# Mustafa: "filter n_ab >= 50 BEFORE computing lift" — applying the absolute
# floor now, not a support fraction that depends on sample size
pair_counts = pair_counts[pair_counts["n_ab"] >= MIN_COUNT].copy()
print(f"[D] {len(pair_counts):,} pairs survive n_ab >= {MIN_COUNT}")

pair_counts["n_nhs_class"] = pair_counts["nhs_bnf_class"].map(nhs_counts)
pair_counts["n_otc_class"] = pair_counts["otc_bnf_class"].map(otc_counts)

pair_counts["support"] = pair_counts["n_ab"] / n_total_baskets
pair_counts["confidence"] = pair_counts["n_ab"] / pair_counts["n_nhs_class"]
pair_counts["lift"] = pair_counts["confidence"] / (pair_counts["n_otc_class"] / n_total_baskets)

[D] 3,549 raw (NHS class, OTC class) pairs found (before MIN_COUNT filter)
[D] 1,624 pairs survive n_ab >= 50


### PART E — Filter to genuinely OTC-eligible consequent classes only

In [7]:
otc_products_bnf = otc_products.merge(
    transactions[["ProductKey", "bnf_class"]].drop_duplicates(), on="ProductKey", how="left"
)
otc_eligible_classes = set(otc_products_bnf["bnf_class"].dropna())

rules = pair_counts[pair_counts["otc_bnf_class"].isin(otc_eligible_classes)].copy()
rules = rules.sort_values(["support", "lift"], ascending=False)
print(f"[E] {len(rules):,} rules with an OTC-eligible consequent class "
      f"(of {len(pair_counts):,} candidate pairs)")

# Same-class check (BNF-level equivalent of the earlier brand-substitution
# filter — flags a class recommending back to itself)
rules["is_same_class"] = rules["nhs_bnf_class"] == rules["otc_bnf_class"]
print(f"[E] {rules['is_same_class'].sum():,} same-class rules (NHS class == OTC class) — "
      f"review these individually; a class recommending itself may still be "
      f"clinically meaningful (e.g. different sub-paragraph) unlike the old "
      f"SKU-level brand-duplicate problem, but is worth a manual look")

[E] 1,624 rules with an OTC-eligible consequent class (of 1,624 candidate pairs)
[E] 28 same-class rules (NHS class == OTC class) — review these individually; a class recommending itself may still be clinically meaningful (e.g. different sub-paragraph) unlike the old SKU-level brand-duplicate problem, but is worth a manual look


### PART F — Attach human-readable BNF descriptions

In [8]:
bnf_hier = None
try:
    from edw_helpers import authenticate, load_table_smart
    authenticate()
    bnf_hier_raw = load_table_smart("Dim_BNFHierarchy", columns=["Chapter", "Section", "Depth", "Description"])
    bnf_hier = bnf_hier_raw[bnf_hier_raw["Depth"] == 2].copy()
    bnf_hier["bnf_class"] = bnf_hier["Chapter"].astype(str) + bnf_hier["Section"].astype(str)
    bnf_hier = bnf_hier.set_index("bnf_class")["Description"]
except Exception as e:
    print(f"[F] Skipping BNF class name lookup ({e}) — will show raw class codes instead")

if bnf_hier is not None:
    rules["nhs_class_name"] = rules["nhs_bnf_class"].map(bnf_hier).fillna(rules["nhs_bnf_class"])
    rules["otc_class_name"] = rules["otc_bnf_class"].map(bnf_hier).fillna(rules["otc_bnf_class"])
else:
    rules["nhs_class_name"] = rules["nhs_bnf_class"]
    rules["otc_class_name"] = rules["otc_bnf_class"]

Stage A  DeviceCodeCredential created
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code GKRW7UFPY to authenticate.
Stage B  datastore resolved
         account_name   = smrttempsa
         container_name = azureml-blobstore-9c0e725a-1db2-4e2b-83ed-97bbde32f29b
Stage C  BlobServiceClient + adlfs filesystem ready


### PART G — Save

In [9]:
rules.to_parquet("bnf_class_otc_rules.parquet", index=False)
rules.to_csv("bnf_class_otc_rules.csv", index=False)

print(f"\nSaved {len(rules):,} BNF-class-level rules -> "
      "bnf_class_otc_rules.parquet / .csv")
print(f"\nDistinct OTC classes recommended: {rules['otc_bnf_class'].nunique()}")
print("\nTop 15 rules by support, then lift:")
print(
    rules[["nhs_class_name", "otc_class_name", "n_ab", "support", "confidence", "lift"]]
    .head(15)
    .to_string(index=False)
)


Saved 1,624 BNF-class-level rules -> bnf_class_otc_rules.parquet / .csv

Distinct OTC classes recommended: 48

Top 15 rules by support, then lift:
                                              nhs_class_name                                               otc_class_name   n_ab  support  confidence      lift
                                      Lipid-regulating drugs                                           Antiplatelet drugs 226796 0.004628    0.045289  3.104868
                            Beta-adrenoceptor blocking drugs                                           Antiplatelet drugs 147549 0.003011    0.086789  5.950007
                              Hypertension and heart failure                                           Antiplatelet drugs  80373 0.001640    0.014827  1.016481
                 Antisecretory drugs and mucosal protectants                                                     Minerals  47182 0.000963    0.014012  1.566606
                 Antisecretory drugs and mucosal pro

In [10]:
print(rules.sort_values("lift", ascending=False).head(20)[
    ["nhs_class_name", "otc_class_name", "n_ab", "support", "confidence", "lift"]
].to_string(index=False))

                                  nhs_class_name                                    otc_class_name  n_ab  support  confidence       lift
                                        Minerals                                             A6000   539 0.000011    0.032643 299.302534
           Drugs affecting intestinal secretions             Drugs affecting intestinal secretions  1849 0.000038    0.041813 270.517106
           Preparations for eczema and psoriasis             Preparations for eczema and psoriasis   296 0.000006    0.003925 100.870417
                         Fluids and electrolytes                           Fluids and electrolytes   101 0.000002    0.044929  94.900535
                                        Minerals                    Drugs acting on the oropharynx   223 0.000005    0.013505  73.435293
Local preparations for anal and rectal disorders  Local preparations for anal and rectal disorders   145 0.000003    0.004100  69.960735
                                  Contrac

In [12]:
import pandas as pd

rules = pd.read_parquet("bnf_class_otc_rules.parquet")

MIN_OTC_CLASS_COUNT = 500   # the consequent class itself must be reasonably
                              # common, or lift against it is statistically unstable
rules_stable = rules[rules["n_otc_class"] >= MIN_OTC_CLASS_COUNT].copy()
print(f"{len(rules) - len(rules_stable)} rules dropped for having too rare a "
      f"consequent class to trust lift against")

print(rules_stable[rules_stable["is_same_class"] == False]
      .sort_values("lift", ascending=False)
      .head(20)
      [["nhs_class_name","otc_class_name","n_ab","n_otc_class","support","confidence","lift"]]
      .to_string(index=False))

3 rules dropped for having too rare a consequent class to trust lift against
                                      nhs_class_name                                    otc_class_name  n_ab  n_otc_class  support  confidence       lift
                                            Minerals                                             A6000   539         5345 0.000011    0.032643 299.302534
                                            Minerals                    Drugs acting on the oropharynx   223         9013 0.000005    0.013505  73.435293
                                    Antifungal drugs        Treatment of vaginal and vulval conditions   411         8195 0.000008    0.009287  55.537934
               Preparations for eczema and psoriasis                  Anti-infective skin preparations  2264        27656 0.000046    0.030021  53.199782
                                            Vitamins                                             A6000   563         5345 0.000011    0.005270  48.322419

In [13]:
transactions = pd.read_parquet("clean_transactions_all_drugs.parquet")

bnf_class = transactions["ProductCodeLevel1"].astype(str) + transactions["ProductCodeLevel2"].astype(str)
print(
    transactions[bnf_class.isin(["A6000", "1102", "0111"])]
    [["ProductCodeLevel1", "ProductCodeLevel2", "DrugName"]]
    .drop_duplicates()
    .head(10)
)

     ProductCodeLevel1 ProductCodeLevel2                                           DrugName
970                 A6               000  Glutafin GF part-baked fibre rolls [DR SCHAR U...
973                 A6               000                   Juvela GF mix [JUVELA LTD] [500]
1763                A6               000  Aymes Shake CHOCOLATE powder (57g sachet) choc...
1793                A6               000                    Aveeno cream [KENVUE UK ] [100]
1884                A6               000              SMA Alfamino powder [NESTLE HS] [400]
2026                A6               000  Fresubin 2kcal vanilla drink vanilla [FRESENIU...
2083                01                11                 SCOPODERM patch 1.5mg [BAXTER] [2]
2276                A6               000  Ensure Plus milkshake style neutral liquid neu...
2799                A6               000         Uvistat SPF50 sun cream [DENDRON BR] [125]
2827                A6               000            Aptamil Pepti 1 powder [NUTR

In [14]:
import pandas as pd

rules = pd.read_parquet("bnf_class_otc_rules.parquet")

MANUAL_BNF_LABELS = {
    "A6000": "ACBS — Nutritional supplements / borderline substances",
}
rules["nhs_class_name"] = rules["nhs_bnf_class"].map(MANUAL_BNF_LABELS).fillna(rules["nhs_class_name"])
rules["otc_class_name"] = rules["otc_bnf_class"].map(MANUAL_BNF_LABELS).fillna(rules["otc_class_name"])

rules.to_parquet("bnf_class_otc_rules.parquet", index=False)
rules.to_csv("bnf_class_otc_rules.csv", index=False)
print("Labels patched and re-saved")

Labels patched and re-saved
